In [1]:
# %%
# Raster Preparation for Final Models 
# Refer to notes for all variables and resampling requirement

import os
import gc
import glob
import numpy as np
import pandas as pd
import xarray as xr
import rioxarray as rxr
from rasterio.enums import Resampling
from pathlib import Path

# Configuration 
START = pd.Timestamp("2001-01-01")
END   = pd.Timestamp("2005-12-31")
 
WET_DAY_THRESHOLD = 0.1   # mm — gridded wet-day definition

# Target resolutions in km
RESOLUTIONS = [1.5, 2, 2.5, 3, 3.5, 4, 4.5, 5, 5.5, 6]
 
DIRS = {
    "era5l_prec":     r"..\Data\Datasets\ERA5L_prec",
    "chirps_prec":    r"..\Data\Datasets\CHIRPS_prec",
    "har_prec":       r"..\Data\Datasets\HAR_prec",
    "persiann_prec":  r"..\Data\Datasets\PERSIANN_prec",
    "mswep_prec":     r"..\Data\Datasets\MSWEP_prec",
    "era5l_wind":     r"..\Data\Datasets\ERA5L_2_wind_speed",
    "era5l_evap":     r"..\Data\Datasets\ERA5L_2_total_evaporation",
    "era5l_pressure": r"..\Data\Datasets\ERA5L_2_surface_pressure",
    "era5l_soil":     r"..\Data\Datasets\ERA5L_2_soil_moisture",
    "era5l_dewpoint": r"..\Data\Datasets\ERA5L_2_dewpoint_temperature",
    "modis_evi":      r"..\Data\Datasets\MODIS_2_EVI"
}

LST_DIRS = {
    "lst_d":    r"C:\Users\A S U S\Documents\Study\ClimateData\CD-Workspace\Data\Datasets\MODIS_2_lst_d",
    "lst_n":    r"C:\Users\A S U S\Documents\Study\ClimateData\CD-Workspace\Data\Datasets\MODIS_2_lst_n",
    "lst_diff": r"C:\Users\A S U S\Documents\Study\ClimateData\CD-Workspace\Data\Datasets\MODIS_2_lst_diff",
}
COMPUTE_LST_DIFF = False

# Build list of (year, month) to process
month_range = pd.period_range(start=START, end=END, freq="M")

# %%
# ── Helpers ───────────────────────────────────────────────────────────────────
 
def resample_to_template(raster: xr.DataArray, template: xr.DataArray) -> xr.DataArray:
    """Reproject and clip raster to match the given template."""
    return raster.rio.reproject_match(template, resampling=Resampling.bilinear)
 
 
def parse_tif_catalog(directory: str) -> pd.DataFrame:
    """
    Build a DataFrame of (filepath, date) for all .tif files in a directory.
    Assumes filenames end with YYYYMMDD.tif (last 8 chars before extension).
    Returns sorted by date.
    """
    files = sorted(glob.glob(os.path.join(directory, "*.tif")))
    if not files:
        raise FileNotFoundError(f"No .tif files found in {directory}")
    rows = []
    for f in files:
        stem = Path(f).stem          # filename without extension
        date_str = stem[-8:]         # last 8 chars = YYYYMMDD
        try:
            date = pd.Timestamp(date_str)
            rows.append({"path": f, "date": date})
        except ValueError:
            continue
    return pd.DataFrame(rows).sort_values("date").reset_index(drop=True)
 
 
def filter_period(catalog: pd.DataFrame, start: pd.Timestamp, end: pd.Timestamp) -> pd.DataFrame:
    return catalog[(catalog["date"] >= start) & (catalog["date"] <= end)].reset_index(drop=True)
 
 
def stack_month(catalog: pd.DataFrame, year: int, month: int, template: xr.DataArray) -> xr.DataArray | None:
    """
    Load all daily rasters for year/month, resample each to template, and stack along
    a new 'day' dimension. Returns None if no files found for that month.
    """
    month_files = catalog[
        (catalog["date"].dt.year == year) &
        (catalog["date"].dt.month == month)
    ]
    if month_files.empty:
        return None
 
    slices = []
    for _, row in month_files.iterrows():
        da = rxr.open_rasterio(row["path"], masked=True)
        da = resample_to_template(da, template)
        # Squeeze band dim if single-band
        if "band" in da.dims and da.sizes["band"] == 1:
            da = da.squeeze("band", drop=True)
        da = da.assign_coords(day=row["date"])
        slices.append(da.load())  # load into memory before closing
        da.close()
 
    stack = xr.concat(slices, dim="day")
    return stack


def stack_month_lst(catalog: pd.DataFrame, year: int, month: int, template: xr.DataArray):
    """
    Load LST daily rasters for year/month resampled to template.
    Returns (stack, n_expected, n_available). stack is None if no files found.
    """
    month_files = catalog[
        (catalog["date"].dt.year  == year) &
        (catalog["date"].dt.month == month)
    ]
    days_in_month = pd.Period(f"{year}-{month:02d}", freq="M").days_in_month
 
    if month_files.empty:
        return None, days_in_month, 0
 
    slices = []
    for _, row in month_files.iterrows():
        da = rxr.open_rasterio(row["path"], masked=True)
        da = resample_to_template(da, template)
        if "band" in da.dims and da.sizes["band"] == 1:
            da = da.squeeze("band", drop=True)
        da = da.assign_coords(day=row["date"])
        slices.append(da.load())
        da.close()
 
    stack = xr.concat(slices, dim="day")
    return stack, days_in_month, len(slices)
 
 
def save_raster(data: xr.DataArray, out_path: str, crs, nodata: float = -9999.0):
    """Write a 2D DataArray to GeoTiff, preserving CRS and nodata."""
    data = data.rio.write_crs(crs)
    data = data.rio.write_nodata(nodata)
    data.rio.to_raster(out_path)
 
 
def month_label(year: int, month: int) -> str:
    return f"{year}{month:02d}"

def expected_composites(year: int, month: int) -> int:
    """Approximate number of 16-day composites expected to overlap a month."""
    days_in_month = pd.Period(f"{year}-{month:02d}", freq="M").days_in_month
    return max(1, round(days_in_month / 16))


def process_lst_product(product: str, directory: str, month_range: pd.PeriodIndex, template: xr.DataArray, lst_out: str):
    """Process one LST product directory → monthly mean + per-pixel missing fraction."""
    if not os.path.isdir(directory):
        print(f"  [{product}] Directory not found: {directory} — skipping")
        return
 
    catalog = filter_period(parse_tif_catalog(directory), START, END)
    if catalog.empty:
        print(f"  [{product}] No files found in period — skipping")
        return
 
    crs = rxr.open_rasterio(catalog["path"].iloc[0]).rio.crs
 
    for period in month_range:
        year, month = period.year, period.month
        label = month_label(year, month)
 
        mean_path    = os.path.join(lst_out, f"{product}_monthly_mean_{label}.tif")
        missing_path = os.path.join(lst_out, f"{product}_missing_fraction_{label}.tif")
 
        if os.path.exists(mean_path) and os.path.exists(missing_path):
            print(f"  [{product}] {label} — skipping (exists)")
            continue
 
        stack, n_expected, n_available = stack_month_lst(catalog, year, month, template)
 
        if stack is None:
            print(f"  [{product}] {label} — WARNING: no files found, skipping")
            continue
 
        # Monthly mean — skipna so valid pixels use whatever days are available
        monthly_mean = stack.mean(dim="day", skipna=True)
        save_raster(monthly_mean, mean_path, crs)
 
        nan_count_in_stack = stack.isnull().sum(dim="day")          
        absent_days        = n_expected - n_available               
        total_nan_days     = nan_count_in_stack + absent_days       
        missing_fraction   = (total_nan_days / n_expected).clip(0, 1)
        missing_fraction   = missing_fraction.astype(np.float32)
        save_raster(missing_fraction, missing_path, crs, nodata=-9999.0)
 
        del stack
        gc.collect()
 
        mean_missing = float(missing_fraction.mean())
        print(f"  [{product}] {label} — {n_available}/{n_expected} days, mean missing fraction={mean_missing:.2f}")

In [3]:

# %%
# ── Main Processing Loop ──────────────────────────────────────────────────────
for res in RESOLUTIONS:
    print(f"\n{'=' * 80}")
    print(f" STARTING PROCESSING FOR {res}KM RESOLUTION")
    print(f"{'=' * 80}\n")
    
    res_str = str(res).replace('.', '_')
    OUTPUT_DIR   = rf"..\Data\Processed\RegressionModel\Monthly_Rasters_{res_str}km"
    TEMPLATE_PATH = rf"..\Data\Datasets\TemplateRasters\template_raster_WGS_{res_str}km.tif"
    
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    # Load specific resolution template
    template = rxr.open_rasterio(TEMPLATE_PATH, masked=True)

    # ── 1. Precipitation products ─────────────────────────────────────────────
    PREC_PRODUCTS = {
        "era5l":    DIRS["era5l_prec"],
        "chirps":   DIRS["chirps_prec"],
        "har":      DIRS["har_prec"],
        "persiann": DIRS["persiann_prec"],
        'mswep':    DIRS['mswep_prec']
    }
     
    print("-" * 60)
    print(f"Processing precipitation products ({res}km)...")
    print("-" * 60)
     
    for product, directory in PREC_PRODUCTS.items():
        product_out = os.path.join(OUTPUT_DIR, product)
        os.makedirs(product_out, exist_ok=True)
     
        catalog = filter_period(parse_tif_catalog(directory), START, END)
        crs = rxr.open_rasterio(catalog["path"].iloc[0]).rio.crs
     
        for period in month_range:
            year, month = period.year, period.month
            label = month_label(year, month)
     
            sum_path = os.path.join(product_out, f"{product}_monthly_sum_{label}.tif")
            
            extra_exist = True
            if product == "chirps":
                wdc_path = os.path.join(product_out, f"chirps_wet_day_count_{label}.tif")
                std_path = os.path.join(product_out, f"chirps_monthly_std_{label}.tif")
                extra_exist = os.path.exists(wdc_path) and os.path.exists(std_path)
            
            if os.path.exists(sum_path) and extra_exist:
                print(f"  [{product}] {label} — skipping (exists)")
                continue
     
            stack = stack_month(catalog, year, month, template)
            if stack is None:
                print(f"  [{product}] {label} — WARNING: no files found, skipping")
                continue
     
            monthly_sum = stack.sum(dim="day", skipna=True)
            save_raster(monthly_sum, sum_path, crs)
     
            if product == "chirps":
                wet_day_count = (stack >= WET_DAY_THRESHOLD).sum(dim="day")
                wet_day_count = wet_day_count.astype(np.float32)
                save_raster(wet_day_count, wdc_path, crs)
     
                monthly_std = stack.std(dim="day", skipna=True)
                save_raster(monthly_std, std_path, crs)
     
            del stack
            gc.collect()
            print(f"  [{product}] {label} — done")



    # ── 3. MODIS EVI (16-day composites) ──────────────────────────────────────
    print("\n" + "-" * 60)
    print(f"Processing MODIS EVI (16-day composites → monthly mean) ({res}km)...")
    print("-" * 60)
     
    evi_out = os.path.join(OUTPUT_DIR, "modis_evi")
    os.makedirs(evi_out, exist_ok=True)
     
    evi_catalog = parse_tif_catalog(DIRS["modis_evi"])
    evi_catalog["end_date"] = evi_catalog["date"] + pd.Timedelta(days=15)
     
    crs_evi = rxr.open_rasterio(evi_catalog["path"].iloc[0]).rio.crs
     
    for period in month_range:
        year, month = period.year, period.month
        label = month_label(year, month)
     
        mean_path    = os.path.join(evi_out, f"evi_monthly_mean_{label}.tif")
        missing_path = os.path.join(evi_out, f"evi_missing_fraction_{label}.tif")
     
        if os.path.exists(mean_path) and os.path.exists(missing_path):
            print(f"  [EVI] {label} — skipping (exists)")
            continue
     
        month_start = pd.Timestamp(year=year, month=month, day=1)
        month_end   = month_start + pd.offsets.MonthEnd(0)
     
        overlapping = evi_catalog[
            (evi_catalog["date"]     <= month_end) &
            (evi_catalog["end_date"] >= month_start)
        ]
     
        n_expected  = expected_composites(year, month)
        n_available = len(overlapping)
        missing_frac = max(0.0, (n_expected - n_available) / n_expected)
     
        if n_available == 0:
            print(f"  [EVI] {label} — WARNING: no composites found, skipping")
            continue
     
        slices = []
        for _, row in overlapping.iterrows():
            da = rxr.open_rasterio(row["path"], masked=True)
            da = resample_to_template(da, template)
            if "band" in da.dims and da.sizes["band"] == 1:
                da = da.squeeze("band", drop=True)
            slices.append(da.load())
            da.close()
     
        stack   = xr.concat(slices, dim="composite")
        evi_mean = stack.mean(dim="composite", skipna=True)
        save_raster(evi_mean, mean_path, crs_evi)
     
        missing_arr = xr.full_like(evi_mean, fill_value=float(missing_frac))
        save_raster(missing_arr, missing_path, crs_evi, nodata=-9999.0)
     
        del stack, slices
        gc.collect()
        print(f"  [EVI] {label} — {n_available}/{n_expected} composites, missing={missing_frac:.2f}")

    # ── 4. MODIS LST (daily files, high missingness) ──────────────────────────
    print("\n" + "-" * 60)
    print(f"Processing MODIS LST (daily, high missingness) ({res}km)...")
    print("-" * 60)
     
    lst_out = os.path.join(OUTPUT_DIR, "modis_lst")
    os.makedirs(lst_out, exist_ok=True)
     
    # Process lst_d and lst_n unconditionally for this resolution
    for product in ["lst_d", "lst_n"]:
        process_lst_product(product, LST_DIRS[product], month_range, template, lst_out)
     
    if COMPUTE_LST_DIFF:
        print("\n  [lst_diff] Computing from lst_d minus lst_n stacks...")
     
        d_catalog = filter_period(parse_tif_catalog(LST_DIRS["lst_d"]), START, END)
        n_catalog = filter_period(parse_tif_catalog(LST_DIRS["lst_n"]), START, END)
        crs = rxr.open_rasterio(d_catalog["path"].iloc[0]).rio.crs
     
        for period in month_range:
            year, month = period.year, period.month
            label = month_label(year, month)
     
            mean_path    = os.path.join(lst_out, f"lst_diff_monthly_mean_{label}.tif")
            missing_path = os.path.join(lst_out, f"lst_diff_missing_fraction_{label}.tif")
     
            if os.path.exists(mean_path) and os.path.exists(missing_path):
                print(f"  [lst_diff] {label} — skipping (exists)")
                continue
     
            stack_d, n_exp_d, n_avail_d = stack_month_lst(d_catalog, year, month, template)
            stack_n, n_exp_n, n_avail_n = stack_month_lst(n_catalog, year, month, template)
     
            if stack_d is None or stack_n is None:
                print(f"  [lst_diff] {label} — WARNING: missing d or n stack, skipping")
                continue
     
            common_days = np.intersect1d(stack_d["day"].values, stack_n["day"].values)
            if len(common_days) == 0:
                print(f"  [lst_diff] {label} — WARNING: no matching d/n days, skipping")
                continue
     
            sd = stack_d.sel(day=common_days)
            sn = stack_n.sel(day=common_days)
            diff_stack = sd - sn 
     
            monthly_mean = diff_stack.mean(dim="day", skipna=True)
            save_raster(monthly_mean, mean_path, crs)
     
            nan_count      = diff_stack.isnull().sum(dim="day")
            absent_days    = n_exp_d - len(common_days)
            total_nan_days = nan_count + absent_days
            missing_frac   = (total_nan_days / n_exp_d).clip(0, 1).astype(np.float32)
            save_raster(missing_frac, missing_path, crs, nodata=-9999.0)
     
            del stack_d, stack_n, diff_stack
            gc.collect()
            print(f"  [lst_diff] {label} — {len(common_days)} matched days")
     
    else:
        # lst_diff pre-computed files exist in their own directory
        process_lst_product("lst_diff", LST_DIRS["lst_diff"], month_range, template, lst_out)
        
    print(f"\n{res}km raster aggregation complete. Outputs in: {OUTPUT_DIR}")
    
    # Close template for this resolution to free memory before next loop
    template.close()
    gc.collect()


 STARTING PROCESSING FOR 1.5KM RESOLUTION

------------------------------------------------------------
Processing precipitation products (1.5km)...
------------------------------------------------------------
  [era5l] 200101 — done
  [era5l] 200102 — done
  [era5l] 200103 — done
  [era5l] 200104 — done
  [era5l] 200105 — done
  [era5l] 200106 — done
  [era5l] 200107 — done
  [era5l] 200108 — done
  [era5l] 200109 — done
  [era5l] 200110 — done
  [era5l] 200111 — done
  [era5l] 200112 — done
  [era5l] 200201 — done
  [era5l] 200202 — done
  [era5l] 200203 — done
  [era5l] 200204 — done
  [era5l] 200205 — done
  [era5l] 200206 — done
  [era5l] 200207 — done
  [era5l] 200208 — done
  [era5l] 200209 — done
  [era5l] 200210 — done
  [era5l] 200211 — done
  [era5l] 200212 — done
  [era5l] 200301 — done
  [era5l] 200302 — done
  [era5l] 200303 — done
  [era5l] 200304 — done
  [era5l] 200305 — done
  [era5l] 200306 — done
  [era5l] 200307 — done
  [era5l] 200308 — done
  [era5l] 200309 — do